# Face Statistics: UHM Generated Data vs Planck Scans vs Actual Dataset

Two separate comparisons:
- **A — Face region**: UHM generated (face-masked) vs Planck scan variants
- **B — Full face model**: Dataset `full_morphed_face.pth` (train/val) vs UHM generated (full head, no mask)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import open3d as o3d
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path
from scipy import stats
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

UHM_DATA_DIR  = Path('/data/datasets/UHM_generated_data/')
MASK_PATH     = '/data/UHM_models/Landmarks and masks/light_model_face_mask.pkl'
PLANCK_BASE   = Path('planck_scans')
FACES_DATA_DIR = Path('data/faces/data')

PLANCK_FOLDERS = {
    'npy':                PLANCK_BASE / 'npy',
    'npy_scaled':         PLANCK_BASE / 'npy_scaled',
    'npy_scaled_uniform': PLANCK_BASE / 'npy_scaled_uniform',
}

## Load UHM face mask

In [ ]:
with open(MASK_PATH, 'rb') as f:
    face_mask = pickle.load(f)  # (V,) bool

print(f'Total head vertices : {len(face_mask):,}')
print(f'Face-region vertices: {face_mask.sum():,}  ({face_mask.mean()*100:.1f}%)')

## Collect statistics

In [ ]:
def bbox_stats(pts: np.ndarray) -> dict:
    mn, mx = pts.min(axis=0), pts.max(axis=0)
    return {
        'width':  float(mx[0] - mn[0]),
        'height': float(mx[1] - mn[1]),
        'depth':  float(mx[2] - mn[2]),
        'n_points': len(pts),
    }

# --- UHM: face-masked AND full head ---
ply_files = sorted(p for p in UHM_DATA_DIR.glob('*.ply') if not p.name.startswith('.'))
print(f'UHM PLY files: {len(ply_files):,}')

uhm_face_records = []
uhm_full_records = []
for path in tqdm(ply_files, desc='UHM'):
    pcd = o3d.io.read_point_cloud(str(path))
    pts = np.asarray(pcd.points)
    uhm_face_records.append(bbox_stats(pts[face_mask]))
    uhm_full_records.append(bbox_stats(pts))

df_uhm_face = pd.DataFrame(uhm_face_records)
df_uhm_full = pd.DataFrame(uhm_full_records)

# --- Planck ---
planck_dfs = {}
for name, folder in PLANCK_FOLDERS.items():
    records = [bbox_stats(np.load(p)) for p in sorted(folder.glob('*.npy'))]
    planck_dfs[name] = pd.DataFrame(records)
    print(f'Planck {name}: {len(records)} scans')

# --- Actual dataset (train / val) ---
dataset_dfs = {}
for split in ['train', 'val']:
    records = []
    for subj_dir in tqdm(sorted((FACES_DATA_DIR / split).iterdir()), desc=f'Dataset {split}'):
        pts = torch.load(subj_dir / 'full_morphed_face.pth',
                         map_location='cpu', weights_only=False)
        if not isinstance(pts, np.ndarray):
            pts = pts.numpy()
        records.append(bbox_stats(pts))
    dataset_dfs[split] = pd.DataFrame(records)
    print(f'Dataset {split}: {len(records)} subjects')

# Summary tables
print('\n=== A — Face region ===')
print('\n--- UHM face (masked) ---')
display(df_uhm_face[['width','height','depth']].describe().round(4))
for name, df in planck_dfs.items():
    print(f'\n--- Planck {name} ---')
    display(df[['width','height','depth']].describe().round(4))

print('\n=== B — Full head model ===')
print('\n--- UHM full head ---')
display(df_uhm_full[['width','height','depth']].describe().round(4))
for split, df in dataset_dfs.items():
    print(f'\n--- Dataset {split} ---')
    display(df[['width','height','depth']].describe().round(4))

---
# A — Face region: UHM face (masked) vs Planck scans

In [ ]:
def plot_histograms(dfs: dict, palette: list, title: str):
    dims = ['width', 'height', 'depth']
    fig, axes = plt.subplots(len(dims), len(dfs), figsize=(len(dfs) * 4, 10), sharey='row')
    if len(dfs) == 1:
        axes = [[ax] for ax in axes]
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for col, (label, df) in enumerate(dfs.items()):
        color = palette[col]
        for row, dim in enumerate(dims):
            ax = axes[row][col]
            data = df[dim]
            ax.hist(data, bins=min(40, len(data)), color=color, edgecolor='white',
                    linewidth=0.5, density=True, alpha=0.8)
            try:
                kde = stats.gaussian_kde(data)
                xs  = np.linspace(data.min(), data.max(), 300)
                ax.plot(xs, kde(xs), 'k-', linewidth=1.2)
            except Exception:
                pass
            ax.axvline(data.mean(),   color='black', lw=1.2, ls='--', label=f'μ={data.mean():.3f}')
            ax.axvline(data.median(), color='grey',  lw=1.0, ls=':',  label=f'med={data.median():.3f}')
            ax.legend(fontsize=7); ax.set_yticks([])
            if row == 0: ax.set_title(label, fontsize=10)
            if col == 0: ax.set_ylabel(dim, fontsize=11)
            ax.set_xlabel('units')
    plt.tight_layout(); plt.show()

def plot_boxplots(dfs: dict, palette: list, title: str):
    dims = ['width', 'height', 'depth']
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    labels = list(dfs.keys())
    for ax, dim in zip(axes, dims):
        bp = ax.boxplot([df[dim].values for df in dfs.values()], patch_artist=True,
                        widths=0.5, medianprops=dict(color='black', linewidth=2))
        for patch, color in zip(bp['boxes'], palette):
            patch.set_facecolor(color); patch.set_alpha(0.8)
        ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
        ax.set_title(dim, fontsize=11); ax.set_ylabel('units')
        ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout(); plt.show()

dims = ['width', 'height', 'depth']

# --- Comparison A ---
face_dfs    = {'UHM face (masked)': df_uhm_face, **{f'Planck {k}': v for k, v in planck_dfs.items()}}
face_palette = ['#E15759', '#4C72B0', '#DD8452', '#55A868']

plot_histograms(face_dfs, face_palette, 'A — Face region: UHM face vs Planck (histograms)')
plot_boxplots(face_dfs, face_palette,   'A — Face region: UHM face vs Planck (box plots)')

---
# B — Full face model: Dataset (train/val) vs UHM full head

In [ ]:
full_dfs     = {'UHM full head': df_uhm_full, 'Dataset train': dataset_dfs['train'], 'Dataset val': dataset_dfs['val']}
full_palette = ['#F28E2B', '#8172B3', '#64B5CD']

plot_histograms(full_dfs, full_palette, 'B — Full face model: UHM full head vs Dataset train/val (histograms)')
plot_boxplots(full_dfs, full_palette,   'B — Full face model: UHM full head vs Dataset train/val (box plots)')

## Example scans — UHM face (masked)

In [ ]:
N_EXAMPLES = 6

fig = plt.figure(figsize=(N_EXAMPLES * 3.5, 10))
fig.suptitle('Example UHM face scans (masked)', fontsize=13, fontweight='bold')

for col, path in enumerate(ply_files[:N_EXAMPLES]):
    pcd  = o3d.io.read_point_cloud(str(path))
    pts  = np.asarray(pcd.points)
    face = pts[face_mask]
    x, y, z = face[:, 0], face[:, 1], face[:, 2]

    ax3d = fig.add_subplot(3, N_EXAMPLES, col + 1, projection='3d')
    ax3d.scatter(x, y, z, s=0.5, c=z, cmap='viridis', alpha=0.6)
    ax3d.set_title(path.stem[:14], fontsize=7)
    ax3d.tick_params(labelsize=5)

    ax_xy = fig.add_subplot(3, N_EXAMPLES, N_EXAMPLES + col + 1)
    ax_xy.scatter(x, y, s=0.3, c=z, cmap='viridis', alpha=0.5)
    ax_xy.set_aspect('equal'); ax_xy.set_title('front XY', fontsize=7)
    ax_xy.tick_params(labelsize=5)

    ax_zy = fig.add_subplot(3, N_EXAMPLES, 2 * N_EXAMPLES + col + 1)
    ax_zy.scatter(z, y, s=0.3, c=x, cmap='plasma', alpha=0.5)
    ax_zy.set_aspect('equal'); ax_zy.set_title('side ZY', fontsize=7)
    ax_zy.tick_params(labelsize=5)

plt.tight_layout()
plt.show()

## Example scans — Planck (one folder per row)

In [ ]:
planck_colors = {'npy': 'viridis', 'npy_scaled': 'plasma', 'npy_scaled_uniform': 'inferno'}

for folder_name, folder_path in PLANCK_FOLDERS.items():
    files = sorted(folder_path.glob('*.npy'))[:N_EXAMPLES]

    fig = plt.figure(figsize=(N_EXAMPLES * 3.5, 10))
    fig.suptitle(f'Example Planck scans — {folder_name}', fontsize=13, fontweight='bold')
    cmap = planck_colors[folder_name]

    for col, p in enumerate(files):
        pts = np.load(p)
        if len(pts) > 5000:
            rng = np.random.default_rng(0)
            pts = pts[rng.choice(len(pts), 5000, replace=False)]
        x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]

        ax3d = fig.add_subplot(3, N_EXAMPLES, col + 1, projection='3d')
        ax3d.scatter(x, y, z, s=0.5, c=z, cmap=cmap, alpha=0.6)
        ax3d.set_title(p.stem[:14], fontsize=7)
        ax3d.tick_params(labelsize=5)

        ax_xy = fig.add_subplot(3, N_EXAMPLES, N_EXAMPLES + col + 1)
        ax_xy.scatter(x, y, s=0.5, c=z, cmap=cmap, alpha=0.5)
        ax_xy.set_aspect('equal'); ax_xy.set_title('front XY', fontsize=7)
        ax_xy.tick_params(labelsize=5)

        ax_zy = fig.add_subplot(3, N_EXAMPLES, 2 * N_EXAMPLES + col + 1)
        ax_zy.scatter(z, y, s=0.5, c=x, cmap=cmap, alpha=0.5)
        ax_zy.set_aspect('equal'); ax_zy.set_title('side ZY', fontsize=7)
        ax_zy.tick_params(labelsize=5)

    plt.tight_layout()
    plt.show()

## Example scans — Actual dataset (train & val)

In [ ]:
split_colors = {'train': '#8172B3', 'val': '#64B5CD'}

for split in ['train', 'val']:
    subj_dirs = sorted((FACES_DATA_DIR / split).iterdir())[:N_EXAMPLES]
    cmap = 'Purples' if split == 'train' else 'Blues'

    fig = plt.figure(figsize=(N_EXAMPLES * 3.5, 10))
    fig.suptitle(f'Example dataset faces — {split}', fontsize=13, fontweight='bold')

    for col, subj_dir in enumerate(subj_dirs):
        pts = torch.load(subj_dir / 'full_morphed_face.pth',
                         map_location='cpu', weights_only=False)
        if not isinstance(pts, np.ndarray):
            pts = pts.numpy()
        x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]

        ax3d = fig.add_subplot(3, N_EXAMPLES, col + 1, projection='3d')
        ax3d.scatter(x, y, z, s=0.5, c=z, cmap='viridis', alpha=0.6)
        ax3d.set_title(subj_dir.name, fontsize=7)
        ax3d.tick_params(labelsize=5)

        ax_xy = fig.add_subplot(3, N_EXAMPLES, N_EXAMPLES + col + 1)
        ax_xy.scatter(x, y, s=0.3, c=z, cmap='viridis', alpha=0.5)
        ax_xy.set_aspect('equal'); ax_xy.set_title('front XY', fontsize=7)
        ax_xy.tick_params(labelsize=5)

        ax_zy = fig.add_subplot(3, N_EXAMPLES, 2 * N_EXAMPLES + col + 1)
        ax_zy.scatter(z, y, s=0.3, c=x, cmap='plasma', alpha=0.5)
        ax_zy.set_aspect('equal'); ax_zy.set_title('side ZY', fontsize=7)
        ax_zy.tick_params(labelsize=5)

    plt.tight_layout()
    plt.show()

---
# C — Views: view_1 and view_2 statistics (train & val)

In [ ]:
# Collect view stats for both splits
view_dfs = {}  # key: e.g. 'train view_1'
for split in ['train', 'val']:
    for view in ['view_1', 'view_2']:
        records = []
        for subj_dir in tqdm(sorted((FACES_DATA_DIR / split).iterdir()),
                             desc=f'{split} {view}'):
            pts = torch.load(subj_dir / f'{view}.pth',
                             map_location='cpu', weights_only=False)
            if not isinstance(pts, np.ndarray):
                pts = pts.numpy()
            r = bbox_stats(pts)
            r['n_points'] = len(pts)
            records.append(r)
        key = f'{split} {view}'
        view_dfs[key] = pd.DataFrame(records)
        print(f'{key}: {len(records)} scans, '
              f'avg pts={view_dfs[key]["n_points"].mean():.0f}')

print('\nSummary tables:')
for key, df in view_dfs.items():
    print(f'\n--- {key} ---')
    display(df[['n_points','width','height','depth']].describe().round(4))

## C1 — Distributions per view (histograms + box plots)

In [ ]:
view_palette = ['#8172B3', '#B39DDB', '#64B5CD', '#90CAF9']

plot_histograms(view_dfs, view_palette,
                'C — Views: width / height / depth (histograms)')
plot_boxplots(view_dfs, view_palette,
              'C — Views: width / height / depth (box plots)')

# Also show point count distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('C — Number of points per view', fontsize=12, fontweight='bold')
for ax, view_name in zip(axes, ['view_1', 'view_2']):
    for split, color in zip(['train', 'val'], ['#8172B3', '#64B5CD']):
        data = view_dfs[f'{split} {view_name}']['n_points']
        ax.hist(data, bins=40, alpha=0.6, color=color, label=split,
                edgecolor='white', linewidth=0.4, density=True)
    ax.set_title(view_name)
    ax.set_xlabel('n_points')
    ax.set_yticks([])
    ax.legend()
plt.tight_layout()
plt.show()

## C2 — Example views (view_1 and view_2 side-by-side per subject)

In [ ]:
N_VIEW_EXAMPLES = 4  # subjects to show per split

for split in ['train', 'val']:
    subj_dirs = sorted((FACES_DATA_DIR / split).iterdir())[:N_VIEW_EXAMPLES]
    n_rows = N_VIEW_EXAMPLES
    # each subject: 2 views × 2 projections (front XY + side ZY) = 4 cols
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, n_rows * 3.5))
    fig.suptitle(f'Example views — {split}  (col: view1 front | view1 side | view2 front | view2 side)',
                 fontsize=11, fontweight='bold')

    for row, subj_dir in enumerate(subj_dirs):
        for v_idx, view in enumerate(['view_1', 'view_2']):
            pts = torch.load(subj_dir / f'{view}.pth',
                             map_location='cpu', weights_only=False)
            if not isinstance(pts, np.ndarray):
                pts = pts.numpy()
            x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
            cmap = 'viridis' if v_idx == 0 else 'plasma'

            # front XY
            ax = axes[row][v_idx * 2]
            ax.scatter(x, y, s=1, c=z, cmap=cmap, alpha=0.6)
            ax.set_aspect('equal')
            ax.tick_params(labelsize=6)
            if row == 0:
                ax.set_title(f'{view} front XY', fontsize=9)
            if v_idx == 0:
                ax.set_ylabel(subj_dir.name, fontsize=8)

            # side ZY
            ax = axes[row][v_idx * 2 + 1]
            ax.scatter(z, y, s=1, c=x, cmap=cmap, alpha=0.6)
            ax.set_aspect('equal')
            ax.tick_params(labelsize=6)
            if row == 0:
                ax.set_title(f'{view} side ZY', fontsize=9)

    plt.tight_layout()
    plt.show()